<a href="https://colab.research.google.com/github/mikael-pintassilgo/portfolio/blob/my-pages/colab-notebooks/Play_Store_Review_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Date of analysis: end of January 2026.

I would like to develop a mobile puzzle game for the Android platform based on Tetris shapes, but not as dynamic, where the user can think and choose the right place.
I would like it to be profitable.

To increase my chances, I analysed the information on the website
https://app.sensortower.com/top-charts?os=android&country=US&device=iphone&category=game_puzzle&date=2026-01-12
and found that the most popular games in this genre are the following:
- Block Blast! - https://app.sensortower.com/overview/com.block.juggle?country=US.
- Block Crush! - https://app.sensortower.com/overview/com.wood.block.sudoku.puzzle.bm?country=US
- Color Block Jam - https://app.sensortower.com/overview/com.GybeGames.ColorBlockJam?country=US
- Color Block: Combo Blast - https://app.sensortower.com/overview/com.puzzlegames.puzzlebrickslegend?country=US

Another game with similar dynamics is
- Unblock me (premium) - https://app.sensortower.com/overview/com.kiragames.unblockme?country=US

To understand what is good and what is bad about these games, I decided to analyse user feedback.

Below is the code I use for quick analysis of feedback on Google Play.



# Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [ ]:
!pip install google-play-scraper pandas scikit-learn transformers torch nltk sentencepiece

In [ ]:
!pip install symspellpy

# Import Libraries

In [ ]:
import pandas as pd

from google_play_scraper import reviews, Sort
from urllib.parse import urlparse, parse_qs

from collections import Counter
from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans

import spacy
from tqdm import tqdm

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.probability import FreqDist
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer

import re
import os

import time
from datetime import datetime

from symspellpy.symspellpy import SymSpell, Verbosity

# Setup Variables

In [ ]:
path = '/content/drive/MyDrive/Colab Notebooks/mobile_games_feedback_analysis/'

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))

# Initialize the Porter Stemmer.
stemmer = PorterStemmer()

lemmatizer = WordNetLemmatizer()

nlp = spacy.load("en_core_web_sm", disable=["ner"]) # Named Entity Recognition is not required

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

# The original file is here: https://github.com/wolfgarbe/SymSpell/blob/master/SymSpell/frequency_dictionary_en_82_765.txt
# I placed here the version that I used: https://github.com/mikael-pintassilgo/portfolio/blob/main/data-sets/mobile_game_play_store_feedback.csv
sym_spell.load_dictionary(path + "frequency_dictionary_en_82_765.txt", term_index=0, count_index=1)

In [ ]:
SYNONYM_MAP = {
    "bad": {"bad", "terrible", "awful", "horrible", "annoying", "irritating"},
    "good": {"good", "great", "nice", "excellent", "fantastic", "amazing", "wonderful", "favorite", "awesome"},
    "crash": {"crash", "freeze", "hang"},
    "ad": {"ad", "ads", "advertising", "advertize", "advertisement"},
    "bug": {"bug", "issue", "problem", "error"},
    "price": {"price", "pricing", "cost"},
    "purchase": {"purchase", "payment", "pay"},
    "ui": {"ui", "interface", "menu"},
    "relax": {"relax", "relaxing", "calm down", "calmdown", "relaxation", "calming", "antistress"},
    "color": {"Color", "colour"},
}

# Defenitions

In [ ]:
def get_game_id(url: str) -> str | None:
    parsed = urlparse(url)
    params = parse_qs(parsed.query)
    return params.get("id", [None])[0]

## Basic Cleaning

In [ ]:
def clean_text(text):
    # all letters are converted to lowercase
    text = text.lower()

    # Let's replace "im" and "i'm" with "I am".
    text = re.sub(r"\bi'?m\b", 'I am', text, flags=re.IGNORECASE)

    # \1 refers to whatever was captured in the first set of parentheses
    # {2,} looks for 2 or more additional repetitions of that exact character
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # tokenisations and deleting stop words.
    text = re.sub(r'[^a-z ]', '', text)
    words = word_tokenize(text)
    tokens = [w for w in words if w.isalnum() and w not in stop_words]
    return " ".join(tokens)

In [ ]:
def normalize_token(token):
    for canonical, variants in SYNONYM_MAP.items():
        if token in variants:
            return canonical
    return token

## Typos

In [ ]:
def correct_text_sym_spell(text):
    corrected = []
    for word in text.split():
        suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        corrected.append(suggestions[0].term if suggestions else word)
    return " ".join(corrected)

## Stemming

In [ ]:
def stem_text(text):
    words = word_tokenize(text)

    # Apply stemmer to every word in the input text.
    stemmed_words = [stemmer.stem(word) for word in words]

    return " ".join(stemmed_words)

## Lemmatization

In [ ]:
def lemmatize_text(text):
    doc = nlp(text)

    # 1. Lemmatized full text
    lemmatized_tokens = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and not token.is_stop
    ]
    lemmatized_text = " ".join(lemmatized_tokens)

    return lemmatized_text

In [ ]:
def lemmatize_phrases(text):
    doc = nlp(text)

    # 2. Lemmatized meaningful phrases (noun chunks)
    lemmatized_phrases = []
    for chunk in doc.noun_chunks:
        phrase_tokens = [
            normalize_token(token.lemma_.lower())
            for token in chunk
            if token.is_alpha and not token.is_stop
        ]

        if len(phrase_tokens) >= 2:
            # remove single-letter words
            _words = [
                w for w in phrase_tokens
                if len(w) > 1
            ]

            if not _words:
                continue

            # remove duplicates while maintaining order
            _words = list(dict.fromkeys(_words))
            # Let's sort the words, then ‘bad game’ and ‘game bad’ will be the same.
            _words.sort()

            if len(_words) >= 2:
              lemmatized_phrases.append(" ".join(_words))

    return lemmatized_phrases


## Make Canonical

In [ ]:
def get_canonical_word(word: str) -> str:
    synsets = wn.synsets(word)

    if not synsets:
        return word

    # берем самый "общий" synset (обычно первый - наиболее частотный)
    best_synset = synsets[0]

    # берем первую lemma как каноническую
    canonical = best_synset.lemmas()[0].name().lower()

    # wordnet может вернуть слова через "_" -> делаем пробел или оставляем как есть
    canonical = canonical.replace("_", " ")

    return canonical

In [ ]:
def make_words_canonical(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    words = re.findall(r"[a-z']+", text.lower())

    result = []
    for w in words:
        canonical = get_canonical_word(w)
        result.append(canonical)

    return " ".join(result)

## Counters

In [ ]:
def get_common_words(param_df, column_name):
  # Join all rows into one big string
  all_text = " ".join(param_df[column_name].dropna().astype(str)).lower()

  # Extract words with 2 or more characters
  words = re.findall(r'\b\w{2,}\b', all_text)

  # Count word frequencies
  word_counts = Counter(words)
  print("Length of the word_counts: ", len(word_counts))

  # Convert to DataFrame and sort
  common_words_df = (
      pd.DataFrame(word_counts.items(), columns=["word", "count"])
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
  )

  return common_words_df

In [ ]:
def get_miningful_phrases(param_df, column_name, number_of_most_common_words = 50):
    # --------------------------------------------------
  # Flatten + count
  # --------------------------------------------------

  all_phrases = [
      phrase
      for phrase_list in df["lemmatized_phrases"]
      for phrase in phrase_list
  ]

  phrase_counts = Counter(all_phrases)
  print("Number of phrases: ", len(phrase_counts))

  # --------------------------------------------------
  # Result table
  # --------------------------------------------------

  most_common_phrases = pd.DataFrame(
      phrase_counts.most_common(number_of_most_common_words),
      columns=["phrase", "count"]
  )

  return most_common_phrases

# Scrape Reviews of Competitor Games - DO NOT DO THIS BLOCK if you want to get the same result, read the first text in this block.

Below is the code for obtaining feedback.
However, the google-play-scraper library, which is used for scraping feedback, cannot obtain data for a specific period; it takes data in batches starting with the most recent ones.
If you want to draw your own conclusions based on current reviews, then execute this block.
If you plan to analyse my conclusions, it is better to use the reviews that were collected on the date of the analysis.
To do this, you need to take the files with reviews here:
https://github.com/mikael-pintassilgo/portfolio/blob/main/data-sets/mobile_game_play_store_feedback.csv

Download them to your directory. The path to the directory must be specified in the path variable, which is declared above in the Setup Variables section.

Then execute all blocks starting with block Clean Data.

To scrape feedback on the Google Play Store, you need an app ID.
Finding a competitor's app ID (package name) is easy and completely free. In most cases, it works.
Open the competitor's game in the Google Play Store. Copy the app ID from the app URL.

**Example:**

https://play.google.com/store/apps/details?id=com.block.juggle

App ID is the value after id=

In this example, it's: **com.block.juggle**

For analysis, place the URL in the COMPETITOR_APPS dictionary in the following section.
The game ID will be obtained automatically.

For this example, I will take two games, but in a real project, it is better to take more in order to analyse more competing games or related genres.
This will allow you to identify features that users like and problems that repel users.
If you enhance the features and reduce the problems, the chances of success may be higher.

In [ ]:
COMPETITOR_APPS = {
  "Block_Blust": "https://play.google.com/store/apps/details?id=com.block.juggle",
  "Block_Crush": "https://play.google.com/store/apps/details?id=com.wood.block.sudoku.puzzle.bm"
}

TARGET_REVIEWS_PER_GAME = 500
TARGET_NUMBER_OF_REVIEWS = TARGET_REVIEWS_PER_GAME * len(COMPETITOR_APPS)

In [ ]:
all_dfs = []

for game_name, app_URL in COMPETITOR_APPS.items():
  app_id = get_game_id(app_URL)
  print(f"🔍 Scraping {game_name} ({app_id})")

  all_reviews = []
  continuation_token = None

  while len(all_reviews) < TARGET_NUMBER_OF_REVIEWS:
    result, continuation_token = reviews(
      app_id,
      lang='en',
      country='us',
      sort=Sort.NEWEST,
      count=200,
      continuation_token=continuation_token
    )
    all_reviews.extend(result)
    print("continuation_token: ", continuation_token)
    if continuation_token is None:
      break
    time.sleep(1)


  print(f"Collected {len(all_reviews)} reviews for {game_name}")
  df_game = pd.DataFrame(all_reviews)
  df_game['game'] = game_name
  all_dfs.append(df_game)

df = pd.concat(all_dfs, ignore_index=True)

In [ ]:
display(df.head())
display(df.tail())

In [ ]:
df.info()

In [ ]:
output_directory = path
df.to_csv("mobile_game_play_store_feedback.csv")

# Clean Data

## Start here if you want to analyse my results.

You can obtain a file with current data by executing the code in the Scrape Reviews of Competitor Games section if you want to analyse current data.
But if you want to analyse the data that I analysed, place the file ‘mobile_game_play_store_feedback.csv’ with feedback in the directory that you specified in the path variable in the Setup Variables section.
The file can be downloaded here: https://github.com/mikael-pintassilgo/portfolio/blob/main/data-sets/mobile_game_play_store_feedback.csv

In [ ]:
feedback_filename = "mobile_game_play_store_feedback.csv"

df = pd.read_csv(os.path.join(path, feedback_filename))

## Clean Data

## Basic Cleaning

In [ ]:
print('Lengh before dropna: ', len(df))

In [ ]:
df = df[['content', 'score', 'thumbsUpCount', 'at', 'appVersion']]
df = df.dropna()
df = df[df['content'].str.len() > 25]

In [ ]:
print('Lengh after dropna: ', len(df))

In [ ]:
# Basic text cleaning
df['clean_text'] = df['content'].apply(clean_text)
df = df[df['clean_text'] != '']

## Typos

In [ ]:
df["clean_text"] = df["clean_text"].astype(str).apply(correct_text_sym_spell)

In [ ]:
print('Lengh after basic text cleaning: ', len(df))

## Lemmatized, Stammed & Canonical Text

In [ ]:
tqdm.pandas()

In [ ]:
df["lemmatized_text"] = (
    df["clean_text"]
    .astype(str)
    .progress_apply(lambda x: pd.Series(lemmatize_text(x)))
)

In [ ]:
df["lemmatized_phrases"] = (
    df["clean_text"]
    .astype(str)
    .progress_apply(lambda x: lemmatize_phrases(x))
)

In [ ]:
df["canonical_text"] = (
    df["lemmatized_text"]
    .astype(str)
    .progress_apply(lambda x: pd.Series(make_words_canonical(x)))
)

In [ ]:
df["canonical_phrases"] = (
    df["canonical_text"]
    .astype(str)
    .progress_apply(lambda x: lemmatize_phrases(x))
)

In [ ]:
df["stemmed_text"] = df["lemmatized_text"].apply(stem_text)

In [ ]:
display(df.head())
display(df.tail())

In [ ]:
df.to_csv(path + "after_clean_data.csv")

In [ ]:
df_phrases_non_empty = df[df['lemmatized_phrases'].apply(lambda x: len(x) >= 1)]
print('Length of df_phrases_non_empty: ', len(df_phrases_non_empty))
print("DataFrame with rows containing one or more lemmatized phrases:")
display(df_phrases_non_empty[['clean_text', 'lemmatized_phrases']].head())
df_phrases_non_empty.to_csv(path+"df_phrases_non_empty.csv")

# Calculations

## Most Common Words - Clean Text & Canonical Text - For Fast Simple Analysis

In [ ]:
common_words_df = get_common_words(df, 'clean_text')

print("\nTop 50 words:")
display(common_words_df.head(50))

In [ ]:
print("====================")
print("== CANONICAL TEXT ==")
print("====================\n")
common_words_df = get_common_words(df, 'canonical_text')

print("\nTop 50 words:")
display(common_words_df.head(50))

In [ ]:
# Let's analyse the feedback that contain the word ‘change’.
display(df[df['canonical_text'].apply(lambda x: True if 'change' in x else False)][['content', 'canonical_text']].head(20).style.set_properties(**{"text-align": "left"}))

Highlits about the 'change' word:
- Don't like that theme was changed without player's request.
- One player likes how the background color changes if you get all the blocks out.
- One player wants to rotate blocks.
- Some players don't like colors.
- Some players don't like that the game does not work without internet connection.

## Most Common Meaningful Words - Lemmatized Text

In [ ]:
vectorizer = CountVectorizer(
    stop_words="english",
    min_df=5,          # ignore rare words
    #max_df=0.8,         # ignore too-common words
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b"
)

X = vectorizer.fit_transform(df["lemmatized_text"])

word_freq = (
    pd.DataFrame({
        "word": vectorizer.get_feature_names_out(),
        "count": X.sum(axis=0).A1
    })
    .sort_values("count", ascending=False)
)


In [ ]:
len(word_freq)

In [ ]:
print("Top 50 words:")
display(word_freq.head(50))

It is worth analysing separately the feedback that mentions the words relax, change, advertise, wifi, and others.

Usually, users do not like very intrusive advertising. And they like it when they can play without the internet (in this case, you can disable advertising by turning off the internet).

If users write that they play the game to relax, for example, while returning from work or after a hard day, then this can be considered a characteristic for which players choose the game.

If I want my game to be suitable for the same target audience, it should not be too difficult, so that players can relax while playing my game, rather than getting stressed.

In [ ]:
# Let's analyse the feedback that contain the word ‘relax’.
display(df[df['lemmatized_text'].apply(lambda x: True if 'relax' in x else False)][['content']].head(20).style.set_properties(**{"text-align": "left"}))

Judging by the reviews, players really do use the game to relax and calm down.
If I want to make a similar game, I need to take this fact into account and try to preserve this effect in the game so as not to lose this part of the audience.
When I analyse feedback from my game, it is also worth paying attention to how often the word ‘relax’ appears in the feedback.

## Most Common Miningful Phrases - Lemmatized Phrases & Canonical Phrases

In [ ]:
most_common_phrases = get_miningful_phrases(df, 'lemmatized_phrases', 50)

print("Most common 50")
print(most_common_phrases)

In [ ]:
most_common_phrases = get_miningful_phrases(df, 'canonical_phrases', 50)

print("Most common 50")
print(most_common_phrases)

It is worth analysing reviews that contain phrases such as ‘bad game’ to identify features that can be changed in your game to make it better than your competitors.

In [ ]:
# Let's analyse the feedback that contain the phrase ‘bad game’.
display(df[df['lemmatized_text'].apply(lambda x: True if 'bad game' in x else False)][['content', 'lemmatized_text', 'lemmatized_phrases']].head(20).style.set_properties(**{"text-align": "left"}))

'bad game' highlights:
- lots of ads.
- unable to exit the game.
- problems with scoring.

In [ ]:
# Let's analyse the feedback that contain the phrase ‘bad experience.
display(df[df['lemmatized_text'].apply(lambda x: True if 'bad experience' in x else False)][['content', 'lemmatized_text', 'lemmatized_phrases']].head(20).style.set_properties(**{"text-align": "left"}))

## Most Common Meaningful Phrases - Statistical Approach - Lemmatized Text

In [ ]:
vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 3),
    min_df=5,
    max_df=0.9
)

X = vectorizer.fit_transform(df["lemmatized_text"])

phrase_freq = (
    pd.DataFrame({
        "phrase": vectorizer.get_feature_names_out(),
        "count": X.sum(axis=0).A1
    })
    .sort_values("count", ascending=False)
)

In [ ]:
print("Numbere of frases: ", len(phrase_freq))
print("Most common ", 50)
phrase_freq.head(50)

This analysis often contains phrases that refer to advertising.
Most likely, there is a lot of advertising in the games being analysed. If I show less advertising in my game, it is possible that my game will receive a higher rating, even if it is no different from the games being analysed.

There is also the expression ‘high score’. Since this game has a mode where you can play ‘endlessly’ to achieve a high score, it is worth analysing this feedback carefully. I will do this further on in the notebook.

## Most Common Meaningful Phrases - Linguistic Approach - Lemmatized Text

In [ ]:
nlp = spacy.load("en_core_web_sm")

def extract_phrases(texts):
    phrases = []
    for doc in nlp.pipe(texts, batch_size=1000):
        for chunk in doc.noun_chunks:
            phrase = chunk.text.lower()
            if len(phrase.split()) > 1:
                phrases.append(phrase)
    return phrases

phrases = extract_phrases(df["lemmatized_text"].dropna())

phrase_counts = Counter(phrases)

most_common_phrases = pd.DataFrame(
    phrase_counts.most_common(50),
    columns=["phrase", "count"]
)

In [ ]:
print("Number of phrases: ", len(phrase_counts))
print("Most common ", 50)
display(most_common_phrases)

It is worth analysing some of the most critical reviews separately, in which players write, for example, about deleting the game.

In [ ]:
# Let's analyse the feedback that contain the word ‘delete’.
pd.set_option("display.max_colwidth", None)
display(df[df['lemmatized_text'].apply(lambda x: True if 'delete' in x else False)][['content']].style.set_properties(**{"text-align": "left"}))

In [ ]:
# Let's analyse the feedback that contain the word ‘uninstal’.
display(df[df['lemmatized_text'].apply(lambda x: True if 'uninstal' in x else False)][['content']].style.set_properties(**{"text-align": "left"}))

Players delete the game for the following reasons:
- Too many advertisements.
- Points are reset during updates or other actions taken by developers.
- Advertising materials for the game do not provide accurate information about what will be in the game if they install it.
- The music cannot be turned off.

Since the game has an endless mode in which points are counted, it is worth checking out what players write about points and leaderboards if these phrases appear at the top.

In [ ]:
# Let's analyse the feedback that contain the phrase ‘high score’.
display(df[df['lemmatized_text'].apply(lambda x: True if 'high score' in x else False)][['content']].style.set_properties(**{"text-align": "left"})
)

Highlights:
- Players like high scores.
- They compete with their loved ones and friends for points.
- Players pay attention if the points are not counted correctly.
- Players get upset if they cannot beat their record for a long time.

It is worth paying close attention to the scoring system and showing players interesting animations when they manage to beat their record.

Similarly, lines containing other frequently occurring phrases can be analysed.

## Sentiment Analysis

In [ ]:
sentiment_model = pipeline("sentiment-analysis") # Uncomment this line if you want to use the defoult settings. And also comment the next line.
#sentiment_model = pipeline(
#    "sentiment-analysis",
#    model="siebert/sentiment-roberta-large-english",
#    tokenizer="siebert/sentiment-roberta-large-english"
#)
#sentiment_model = pipeline(
#    "sentiment-analysis",
#    model="siebert/sentiment-roberta-large-english",
#    tokenizer="siebert/sentiment-roberta-large-english",
#    device=0  # set -1 if no GPU
#)

In [ ]:
sentiment_start = datetime.now()

sentiments = sentiment_model(
    df['lemmatized_text'].tolist(),
    batch_size=32,
    truncation=True
)

df["sentiment"] = [s["label"] for s in sentiments]
df["sentiment_score"] = [
    s["score"] if s["label"] == "POSITIVE" else -s["score"]
    for s in sentiments
]

sentiment_end = datetime.now()
print(f"Sentiment analysis took {sentiment_end - sentiment_start}")

In [ ]:
df[['sentiment', 'lemmatized_text', 'sentiment_score']].head()

In [ ]:
df['sentiment'].value_counts()

In [ ]:
propotions = df['sentiment'].value_counts(normalize=True)
print(propotions)

Information about the distribution of positive and negative reviews can be used as a benchmark for your own game.
If you have more positive reviews, it means that you have achieved better results than your competitors.
If you have fewer, then you have room for growth. Continue to analyse and experiment.

In [ ]:
# top positive and negative feedback
top_positive = df.sort_values('sentiment_score', ascending=False)[['content', 'sentiment', 'sentiment_score', "lemmatized_text"]].head(10)
top_negative = df.sort_values('sentiment_score', ascending=True)[['content', 'sentiment', 'sentiment_score', "lemmatized_text"]].head(10)

print('Top positive feedback:')
display(
    top_positive[['sentiment_score', "content", "lemmatized_text"]]
        .style
        .set_properties(**{"text-align": "left"})
)
print('\nTop negative feedback:')
display(
    top_negative[['sentiment_score', "content", "lemmatized_text"]]
        .style
        .set_properties(**{"text-align": "left"})
)

In [ ]:
df.to_csv(path + "after_sentiment_analysis.csv")

## Phareses Analiser: Sentiment Score and TF-IDF and Frequency

In [ ]:
# --------------------------------------------------
# 4. TF-IDF for phrases
# --------------------------------------------------

tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 3)
)

In [ ]:
# ==================================================
# Main pipeline
# ==================================================
def analyze_reviews(df):

    # ---- Phrase extraction ----
    phrase_rows = []
    for idx, phrases in df["lemmatized_phrases"].items():
        for phrase in phrases:
            #print(idx, phrases)
            phrase_rows.append({"doc_id": idx, "phrase": phrase})

    phrase_df = pd.DataFrame(phrase_rows)

    phrase_df["sentiment"] = phrase_df["doc_id"].map(df["sentiment_score"])
    tfidf_matrix = tfidf.fit_transform(phrase_df["phrase"])
    phrase_df["tfidf"] = tfidf_matrix.max(axis=1).toarray().flatten()

    # ---- Aggregate phrases ----
    final = (
        phrase_df
        .groupby("phrase")
        .agg(
            frequency=("phrase", "count"),
            avg_tfidf=("tfidf", "mean"),
            avg_sentiment=("sentiment", "mean")
        )
        .reset_index()
    )

    # ---- Importance ranking ----
    final["importance"] = (
        final["frequency"]
        * final["avg_tfidf"]
        * final["avg_sentiment"].abs()
    )

    final = final.sort_values("importance", ascending=False)

    return final

In [ ]:
# ==================================================
# Run
# ==================================================

final = analyze_reviews(df)

In [ ]:
final["avg_sentiment"].hist(bins=30)

In [ ]:
neg_threshold = final["avg_sentiment"].quantile(0.15)
pos_threshold = final["avg_sentiment"].quantile(0.85)

print('neg_threshold: ', neg_threshold)
print('pos_threshold: ', pos_threshold)

In [ ]:
# ---- Split praise vs complaints ----
top_complaints = final[final["avg_sentiment"] < neg_threshold]
top_praises    = final[final["avg_sentiment"] > pos_threshold]

In [ ]:
print("\n🔥 TOP COMPLAINTS")
print(top_complaints.head(20)[
    ["phrase", "frequency", "avg_sentiment"]
])

print("\n✨ TOP PRAISES")
print(top_praises.head(20)[
    ["phrase", "frequency", "avg_sentiment"]
])


In [ ]:
# Let's analyse the feedback that contain the word ‘error’.
display(df[df['lemmatized_text'].apply(lambda x: True if 'error' in x else False)][['content']].head(20).style.set_properties(**{"text-align": "left"}))

In [ ]:
final.head(10)

In [ ]:
final.tail(10)

## Reviewing the raw text


I also selectively reviewed the feedback and found that users sometimes mention the colors used in the app.

I decided to find and review all reviews that mentioned the word "color".

In [ ]:
# Let's analyse the feedback that contain the word ‘color’.
display(df[df['lemmatized_text'].apply(lambda x: True if 'color' in x else False)][['content', 'clean_text']].head(20).style.set_properties(**{"text-align": "left"}))

'color' highlits:
- There are players who like the colour scheme in the game.
- But there are also players who ask for customisation.

# Findings

**Distribution of positive and negative reviews**

Feedback contains 64% positive reviews and 36% negative reviews. When developing a new game, I need to try to increase the percentage of positive reviews to look better than my competitors.

**Relax and Calm Down**

Judging by the reviews, players really do use the game to relax and calm down. If I want to make a similar game, I need to take this fact into account and try to preserve this effect in the game so as not to lose this part of the audience.
For example, do not add narrative part to this game.

When I analyse feedback from my game, it is also worth paying attention to how often the word ‘relax’ appears in the feedback.

**Bad game**

Users say the game is bad for the following reasons:
- Lots of ads.
- Unable to exit the game.
- Problems with scoring.

Since I am also planning to monetise through advertising, I need to analyse in more detail what users don't like, so that there is a balance between playing time and viewing ads, keeping users happy while still making the game profitable.

It may be worth making the exit from the game explicit by placing the button in a prominent place. Although this is not a common pattern in mobile casual games.

Information about high scores is provided below in this section.

**Delete or Uninstall**

Players delete the game for the following reasons:
- Too many advertisements.
- Points are reset during updates or other actions taken by developers.
- Advertising materials for the game do not provide accurate information about what will be in the game if they install it.
- The music cannot be turned off.

Players should be able to turn off background music and other sounds that may be in the game.

**High Score**

Highlights:
- One of the important goals for players is to score as many points as possible.
- They compete with their loved ones and friends for points.
- Players pay attention if the points are not counted correctly.
- Players get upset if they cannot beat their record for a long time.

It is worth paying close attention to the scoring system and showing players interesting animations when they manage to beat their record.

When releasing updates, special care must be taken to ensure that they do not reset or distort players' high scores.

**Color**

- There are players who like the colour scheme in the game.
- But there are also players who ask for customisation.
- There was a player who did not like that theme was changed without player's request.

It is likely that part of the audience could be attracted if the game offered the option to choose the colour of the shapes, first and foremost, and highlighted this option in marketing materials.

**More**
- Block Rotation.
- Work without internet connection.

# Additional work

The approach described above allows you to quickly see the main problems that players write about in their feedback, as well as what they love or praise about the game.

You can then conduct a more in-depth analysis and manually analyse the feedback, assigning categories manually in a separate column.

Alternatively, you can set tags in the first few hundred lines of feedback and train a model on them, which can then predict categories on its own.

You can use the category information to calculate which categories occur more frequently.
It is best to focus your efforts on the most popular categories first, as they affect a larger number of users.

P.S.
Google's data contains release information. If you want to analyze information about a specific release, such as the latest release, you should add a filter by the "appVersion" column.